# mpu

- megatron.core import parallel_state as mpu 是 NVIDIA 大规模分布式训练框架 Megatron-LM 中最核心、最基础的导入语句之一。
- 简单来说，mpu（全称 Model Parallelism Utilities，模型并行工具）是 Megatron-LM 的“分布式神经系统”。
- <font color='red'>它负责管理所有的分布式并行策略（如张量并行、流水线并行、数据并行等），协调成千上万张 GPU 之间的通信与协作。</font>
- https://docs.nvidia.com/megatron-core/developer-guide/latest/apidocs/core/core.parallel_state.html#module-core.parallel_state

以下是关于 mpu 的详细介绍与核心逻辑拆解：
## 🧩 1. 什么是 parallel_state (mpu)？

- 在大规模模型训练中，我们不能把整个模型塞进一张显卡，因此需要将模型和数据拆分到成百上千张 GPU 上。
- <font color='red'>mpu 就是用来建立、记录和管理这些 GPU 分组（进程组）的核心模块。</font>

当你执行 import parallel_state as mpu 时，你实际上引入了一个 <font color='red'>全局的状态管理器</font>。它内部维护了所有 GPU 的“组织架构”，告诉程序：
- 哪些 GPU 是一起计算同一个矩阵切片的（<font color='red'>张量并行组</font>）。
- 哪些 GPU 是前后接力计算不同网络层的（<font color='red'>流水线并行组</font>）。
- 哪些 GPU 是拿着相同模型副本处理不同数据的（<font color='red'>数据并行组</font>）。

## 🗺️ 2. mpu 的核心功能：五大并行维度的“交通指挥官”
- mpu 负责协调 Megatron-LM 中五个正交的并行维度，确保它们互不干扰且高效协作：

| 并行策略 | 缩写 | mpu 的职责 |
| :--- | :--- | :--- |
| 张量并行 | TP | <font color='red'>将模型的单层（如巨大的 Attention 矩阵）切分到多张卡上计算</font>。`mpu` 负责管理这些卡之间的高频通信（All-Reduce）。 |
| 流水线并行 | PP | <font color='green'>将模型的不同层分配给不同的卡</font>。`mpu` 负责管理卡与卡之间激活值和梯度的传递。 |
| 数据并行 | DP | <font color='red'>复制多份模型副本，处理不同批次的数据</font>。`mpu` 负责在训练完一轮后，同步所有副本的梯度。 |
| 专家并行 | EP | 针对 MoE（混合专家）模型，将不同的“专家”网络分配到不同卡上。 |
| 上下文并行 | CP | <font color='red'>针对超长文本序列，将序列切分给不同卡处理</font>（通常与序列并行 SP 结合）。 |

## ⚙️ 3. 核心工作流程：从初始化到调用

### 第一步：初始化“世界地图” (initialize_model_parallel)
- 在训练脚本启动的最开始，mpu 会被初始化。<font color='green'>你需要告诉它总共有多少张卡，以及每种并行策略的规模（比如 TP=8, PP=4）。</font>
    - mpu 会根据这些参数，利用数学算法（如前缀积和秩映射）自动计算出每一张 GPU 的“身份ID”（Global Rank），并将它们划分到不同的通信组（Process Groups）中。

In [ ]:
# 伪代码示例：初始化并行环境
mpu.initialize_model_parallel(
    tensor_model_parallel_size=8,   # 每8张卡切分一个模型层
    pipeline_model_parallel_size=4, # 模型深度切分成4段
    data_parallel_size=16           # 复制16份模型副本
)

### 第二步：提供全局“通讯录” (Getters)
- 初始化完成后，训练过程中的任何代码都可以随时向 mpu 询问当前的分布式状态。
- 你在之前的代码片段中看到的 <font color='red'>mpu.get_tensor_model_parallel_group()</font> 就是典型的用法：

- mpu.get_tensor_model_parallel_rank(): 返回当前显卡在 TP 组里的编号（比如 0 到 7）。
- mpu.get_tensor_model_parallel_world_size(): 返回 TP 组的总卡数。
- mpu.get_data_parallel_group(): 返回当前显卡所属的数据并行通信组句柄，用于后续做梯度同步。

## 💡 4. 为什么它至关重要？
- 在没有 mpu 的情况下，如果你想写一个分布式训练程序，你需要手动去计算：“我是第 53 号卡，我的张量并行伙伴是 52、54、55 号卡，我要跟它们通信...” 这在千卡集群中是几乎不可能维护的噩梦。
- <font color='green'>mpu 将这些复杂的分布式逻辑完全封装了起来</font>。对于上层的模型代码（比如写一个 Transformer 层）来说，它只需要简单地调用 mpu 的接口，就可以无感地在分布式环境下执行矩阵乘法或梯度同步，极大地降低了大规模模型训练的代码复杂度。

### 张量并行 TP
- self.train_tp_size = mpu.get_tensor_model_parallel_world_size()
- self.train_tp_rank = mpu.get_tensor_model_parallel_rank()
- self.train_tp_group = mpu.get_tensor_model_parallel_group()

### 专家并行 EP
- self.train_ep_size = mpu.get_expert_model_parallel_world_size()
- self.train_ep_rank = mpu.get_expert_model_parallel_rank()
- self.train_ep_group = mpu.get_expert_model_parallel_group()

### 专家张量并行 ETP
- self.train_etp_size = mpu.get_expert_tensor_parallel_world_size()
- self.train_etp_rank = mpu.get_expert_tensor_parallel_rank()
- self.train_etp_group = mpu.get_expert_tensor_parallel_group()

### 判断是否需要重分片（Reshard）
- self.need_tp_reshard = self.train_tp_size != self.infer_tp_size

- 作用：如果训练用了 8 卡 TP，而推理只用了 2 卡 TP，<font color='red'>那么两者的权重切分方式完全对不上</font>。这个布尔值就是后续逻辑的“总开关”：
    - 如果为 True，<font color='red'>管理器就必须启动复杂的权重收集、合并、再切分流程</font>；
    - 如果为 False，<font color='red'>则可以直接共享或拷贝，省去大量通信开销</font>。
    
### 判断训练并行度是否大于推理并行度
- self.train_tp_larger = self.train_tp_size > self.infer_tp_size

    - 作用：这决定了权重的“流动方向”和计算策略。
    - 如果是 True（训练切得更碎），<font color='red'>管理器需要把分散在多张卡上的小份权重收集（Gather）起来，再重新切分成推理需要的大份。</font>
    - 如果是 False（推理切得更碎），<font color='red'>管理器则需要把训练的大份权重切分（Split）并广播给更多的推理卡。</font>